# Bag-of-Words & TF-IDF — from scratch

Turning text into vectors, the oldest and most stubbornly useful way. This notebook builds the **bag-of-words** representation, **TF-IDF** weighting, **cosine** retrieval, and **BM25** from scratch on a tiny 3-document corpus, and *proves* each qualitative claim with an `assert` before printing the number — so every figure on the page is measured, not asserted.

Every function and constant is imported from the sibling script [`bow_tfidf.py`](bow_tfidf.py) — the single source of truth that the concept page and the figure generator also import, so nothing here can drift from the prose. The math is pure NumPy and **deterministic**: BoW counts, IDF, TF-IDF, cosine, and BM25 are exact functions of the corpus, so there is nothing stochastic to seed and the same corpus always yields the same vectors on any machine.

## 0. Setup and device honesty

This chapter is pure-Python / NumPy, so it runs on **CPU** with no accelerator — we print the device honestly. There are no random draws anywhere, so reproducibility is automatic.

In [1]:
import numpy as np

# Canonical source of truth: every function and constant comes from the sibling script.
from bow_tfidf import (
    CORPUS, BM25_EXTRA_DOCS, QUERY, TOKEN_PATTERN,
    BM25_K1, BM25_B, TF_DEMO_COUNTS,
    tokenize, build_vocabulary, count_matrix, document_frequency,
    smoothed_idf, plain_idf, tfidf_matrix, cosine_similarity_matrix,
    log_normalized_tf, tfidf_query_scores, bm25_scores, bm25_tf_component,
)

print('device: cpu (pure-Python/numpy)')
print('numpy:', np.__version__)
try:
    import torch
    print('torch:', torch.__version__)
except ImportError:
    print('torch: not importable (not needed — this chapter is pure NumPy)')

device: cpu (pure-Python/numpy)
numpy: 2.4.6


torch: 2.12.0


## 1. The corpus and the bag-of-words matrix

Three short documents. D1 and D2 share the whole sentence frame `the ... sat on the ...`; D3 shares only `cat` and `the` with D1. We **fix a vocabulary** (the sorted set of distinct tokens) and count each word per document. The result is the **document-term matrix**: row = document, column = term, entry = raw count. The name *bag* is the point — order is discarded, only the tally survives.

In [2]:
vocab = build_vocabulary(CORPUS)
counts = count_matrix(CORPUS, vocab)

# The bag forgets order: 'the cat sat on the mat' and 'the mat sat on the cat' must be identical.
assert (count_matrix(('the cat sat on the mat',), vocab)
        == count_matrix(('the mat sat on the cat',), vocab)).all(), 'BoW should be order-invariant'

print('vocabulary:', vocab)
print('|V| =', len(vocab))
print('\nBoW counts (rows = D1, D2, D3):')
print(counts)
# D1 row: cat,mat,on,sat once; 'the' twice; everything else zero.

vocabulary: ['cat', 'chased', 'dog', 'happy', 'log', 'mat', 'on', 'sat', 'the']
|V| = 9

BoW counts (rows = D1, D2, D3):
[[1 0 0 0 0 1 1 1 2]
 [0 0 1 0 1 0 1 1 2]
 [1 1 1 1 0 0 0 0 2]]


## 2. Raw counts are a bad weight — term-frequency variants

Relevance is **concave** in count: the 1st occurrence of a word says a lot, the 50th says little. Raw count is linear and never saturates. The **log-normalized** TF, $1+\ln f$, encodes diminishing returns directly. We check the claim numerically: the raw spread across counts 1, 5, 50 is 50×; the log-normalized spread is far smaller.

In [3]:
raw = np.array(TF_DEMO_COUNTS, dtype=float)
log_norm = np.array([log_normalized_tf(c) for c in TF_DEMO_COUNTS])

raw_spread = raw.max() / raw.min()
log_spread = log_norm.max() / log_norm.min()

# The whole point of a concave TF: it compresses the dynamic range of counts.
assert log_spread < raw_spread, 'log-normalized TF must compress the count range'

for c, ln in zip(TF_DEMO_COUNTS, log_norm):
    print(f'count {c:>2} -> raw tf {float(c):>4.0f} | log-normalized 1+ln f = {ln:.3f}')
print(f'\nraw spread:            {raw_spread:.1f}x')
print(f'log-normalized spread: {log_spread:.2f}x   (concave -> much tighter)')

count  1 -> raw tf    1 | log-normalized 1+ln f = 1.000
count  5 -> raw tf    5 | log-normalized 1+ln f = 2.609
count 50 -> raw tf   50 | log-normalized 1+ln f = 4.912

raw spread:            50.0x
log-normalized spread: 4.91x   (concave -> much tighter)


## 3. Inverse document frequency = the surprisal of a term

TF says how important a term is *within* a document; it says nothing about whether the term **discriminates** *across* the collection. `the` is everywhere and tells you nothing. **IDF** down-weights common terms and up-weights rare ones. The textbook form $\mathrm{idf}=\log(N/df)$ is exactly the **self-information** $-\log p_t$ of the event 'a random document contains this term' — a term in every document carries 0 bits. We verify both the monotonicity and the zero.

In [4]:
df = document_frequency(counts)
n_docs = len(CORPUS)
plain = plain_idf(df, n_docs)      # textbook log(N/df)
smooth = smoothed_idf(counts)      # sklearn log((1+N)/(1+df)) + 1

the_idx = vocab.index('the')       # 'the' is in all 3 docs -> df = N
happy_idx = vocab.index('happy')   # 'happy' is in 1 doc      -> df = 1

# Plain IDF of a term in EVERY document is exactly 0 (log(N/N)); a rarer term is strictly larger.
assert np.isclose(plain[the_idx], 0.0), "plain idf('the') should be 0 — it carries no information"
assert plain[happy_idx] > plain[the_idx], 'a rarer term must have higher IDF'
# Smoothing keeps 'the' strictly positive (1.0) instead of annihilating it.
assert np.isclose(smooth[the_idx], 1.0), "smoothed idf('the') should be exactly 1.0"

print('term     df  plain log(N/df)  smoothed (sklearn)')
for t in vocab:
    j = vocab.index(t)
    print(f'{t:<8} {df[j]:>2}   {plain[j]:>12.4f}   {smooth[j]:>12.4f}')

term     df  plain log(N/df)  smoothed (sklearn)
cat       2         0.4055         1.2877
chased    1         1.0986         1.6931
dog       2         0.4055         1.2877
happy     1         1.0986         1.6931
log       1         1.0986         1.6931
mat       1         1.0986         1.6931
on        2         0.4055         1.2877
sat       2         0.4055         1.2877
the       3         0.0000         1.0000


## 4. TF-IDF = TF × IDF, then L2-normalize — and it matches scikit-learn exactly

Multiply each raw count by its term's IDF, then **L2-normalize** each row to unit length (so document length stops biasing similarity). With raw-count TF, smoothed IDF, and L2 norm, the by-hand matrix reproduces `scikit-learn`'s `TfidfVectorizer` to **machine precision** — the proof that we understand exactly what the library computes.

In [5]:
tfidf_hand = tfidf_matrix(CORPUS, vocab)

# Every row is L2-normalized -> unit length.
row_norms = np.linalg.norm(tfidf_hand, axis=1)
assert np.allclose(row_norms, 1.0), 'each TF-IDF row must be a unit vector after L2 normalization'

from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer(token_pattern=TOKEN_PATTERN, norm='l2')
tfidf_sklearn = tv.fit_transform(CORPUS).toarray()

# Same vocabulary order? sklearn sorts terms too, so the columns line up.
assert list(tv.get_feature_names_out()) == vocab, 'vocabulary order must match sklearn'
max_diff = float(np.abs(tfidf_hand - tfidf_sklearn).max())
assert np.allclose(tfidf_hand, tfidf_sklearn, atol=1e-9), 'by-hand TF-IDF diverged from sklearn'

print('hand == sklearn:', np.allclose(tfidf_hand, tfidf_sklearn, atol=1e-9),
      '| max abs diff:', f'{max_diff:.2e}')
print('\nD1 TF-IDF row (L2-normalized):')
for t, w in zip(vocab, tfidf_hand[0]):
    if w > 0:
        print(f'  {t:<6} {w:.4f}')

hand == sklearn: True | max abs diff: 0.00e+00

D1 TF-IDF row (L2-normalized):
  cat    0.3742
  mat    0.4920
  on     0.3742
  sat    0.3742
  the    0.5812


## 5. Which terms get up- or down-weighted vs raw counts

The headline TF-IDF behaviour, made explicit for D1: the most *frequent* word (`the`, count 2) is **not** the most *informative*. Because `the` is in every document its smoothed IDF is just 1.0 (it carries no discriminating signal), while `mat` — unique to D1 — is lifted by its high IDF above the shared `cat`/`on`/`sat`. We assert that TF-IDF ranks the **distinctive** term `mat` above the **shared** term `cat`, the opposite of what raw counts (tied at 1) would say.

In [6]:
idf = smoothed_idf(counts)
weighted = counts * idf  # un-normalized tf-idf, so the up/down-weighting is on a comparable scale

mat_idx = vocab.index('mat')   # unique to D1 (df = 1) -> high idf
cat_idx = vocab.index('cat')   # shared D1 & D3 (df = 2) -> lower idf

# Raw counts can't tell mat and cat apart in D1 (both appear once)...
assert counts[0, mat_idx] == counts[0, cat_idx] == 1, 'mat and cat both appear once in D1'
# ...but TF-IDF promotes the distinctive term above the shared one.
assert weighted[0, mat_idx] > weighted[0, cat_idx], 'TF-IDF must rank the unique term above the shared one'

print('D1: raw count vs un-normalized TF-IDF')
print(f"{'term':<6} {'raw':>4} {'tf-idf':>8}")
for t in ('the', 'mat', 'cat', 'on', 'sat'):
    j = vocab.index(t)
    print(f'{t:<6} {counts[0, j]:>4} {weighted[0, j]:>8.4f}')
print("\nmat (unique, df=1) outscores cat (shared, df=2) despite equal raw counts.")

D1: raw count vs un-normalized TF-IDF
term    raw   tf-idf
the       2   2.0000
mat       1   1.6931
cat       1   1.2877
on        1   1.2877
sat       1   1.2877

mat (unique, df=1) outscores cat (shared, df=2) despite equal raw counts.


## 6. Cosine similarity — angle, not distance

With each document a unit vector, **cosine similarity** is just the dot product, measuring the *angle* between term profiles — length-invariant, which is exactly the invariance text retrieval needs. We compute the full pairwise matrix and verify the intuition: D1 is more similar to **D2** (shared sentence frame) than to **D3** (shares only `cat`, `the`).

In [7]:
cos = cosine_similarity_matrix(tfidf_hand)

# Self-similarity is 1; the matrix is symmetric; D1 closer to D2 than to D3.
assert np.allclose(np.diag(cos), 1.0), 'a document is identical to itself (cosine 1)'
assert np.allclose(cos, cos.T), 'cosine similarity is symmetric'
assert cos[0, 1] > cos[0, 2], 'D1 should be more similar to D2 (shared frame) than to D3'

print('cosine similarity matrix:')
print('        D1     D2     D3')
for i, lbl in enumerate(('D1', 'D2', 'D3')):
    print(lbl, ' '.join(f'{cos[i, j]:.3f}' for j in range(3)))
print(f'\ncos(D1,D2) = {cos[0,1]:.3f}  >  cos(D1,D3) = {cos[0,2]:.3f}')

cosine similarity matrix:
        D1     D2     D3
D1 1.000 0.618 0.455
D2 0.618 1.000 0.455
D3 0.455 0.455 1.000

cos(D1,D2) = 0.618  >  cos(D1,D3) = 0.455


## 7. Retrieval: TF-IDF reranks better than raw counts

Now the payoff — search. Given the query **`happy cat`**, rank documents by cosine TF-IDF. Crucially we use the **correct fit/transform discipline**: the IDF is learned from the *corpus only*, and the query is projected into that fitted space (fitting on the query would leak its statistics). We contrast TF-IDF retrieval against **raw-count** retrieval and show TF-IDF puts the right document (D3, which matches the *rare* word `happy`) clearly on top, while raw counts are dominated by the ubiquitous `the`.

In [8]:
full_corpus = CORPUS + BM25_EXTRA_DOCS
labels = [f'D{i+1}' for i in range(len(full_corpus))]

# --- TF-IDF retrieval (IDF from corpus, query projected in — no leak) ---
tfidf_scores = tfidf_query_scores(full_corpus, QUERY)

# --- raw-count retrieval, for contrast: cosine of query counts vs each doc's counts ---
fc_vocab = build_vocabulary(full_corpus)
fc_counts = count_matrix(full_corpus, fc_vocab).astype(float)
q_counts = count_matrix((QUERY,), fc_vocab).astype(float)[0]
raw_norms = np.linalg.norm(fc_counts, axis=1); raw_norms[raw_norms == 0] = 1.0
qn = np.linalg.norm(q_counts) or 1.0
raw_scores = (fc_counts / raw_norms[:, None]) @ (q_counts / qn)

best_tfidf = int(np.argmax(tfidf_scores))
# TF-IDF must rank D3 (matches the rare 'happy') first; it weights 'happy' far above 'the'.
assert labels[best_tfidf] == 'D3', 'TF-IDF should rank D3 (matches rare "happy") first'
# TF-IDF separates D3 from D1 more sharply than raw counts do (rare-term reward).
tfidf_gap = tfidf_scores[2] - tfidf_scores[0]
raw_gap = raw_scores[2] - raw_scores[0]
assert tfidf_gap > raw_gap, 'TF-IDF should widen the D3-over-D1 margin vs raw counts'

print(f'query: {QUERY!r}\n')
print(f"{'doc':<4} {'raw-count cos':>14} {'tf-idf cos':>12}  text")
for i in range(len(full_corpus)):
    print(f'{labels[i]:<4} {raw_scores[i]:>14.3f} {tfidf_scores[i]:>12.3f}  {full_corpus[i]!r}')
print(f'\nTF-IDF D3-over-D1 gap {tfidf_gap:.3f}  >  raw-count gap {raw_gap:.3f}  (rare-term reward)')

query: 'happy cat'

doc   raw-count cos   tf-idf cos  text
D1            0.250        0.247  'the cat sat on the mat'
D2            0.000        0.000  'the dog sat on the log'
D3            0.500        0.615  'the happy cat chased the dog'
D4            0.000        0.000  'a quick brown fox jumps over the lazy dog'
D5            0.000        0.000  'cats and dogs are common household pets'

TF-IDF D3-over-D1 gap 0.368  >  raw-count gap 0.250  (rare-term reward)


## 8. BM25 — the ranking function search engines actually use

TF-IDF cosine is a fine baseline, but production search ranks with **BM25**, which adds two things: term-frequency **saturation** (knob $k_1$) so a keyword-stuffed document can't score arbitrarily high, and document-length **normalization** (knob $b$). First we verify the saturation property directly — BM25's TF component approaches the ceiling $k_1+1$ — then we rank the corpus and confirm BM25 agrees with TF-IDF on order (D3 > D1) while widening the margin.

In [9]:
# Saturation: the TF component climbs toward, but never reaches, the ceiling k1 + 1.
comps = [bm25_tf_component(f, k1=BM25_K1) for f in (1, 2, 5, 10, 50, 1000)]
ceiling = BM25_K1 + 1
assert all(c < ceiling for c in comps), 'BM25 TF must stay below its ceiling k1+1'
assert comps[-1] > comps[0], 'BM25 TF is still monotonically increasing in count'
assert (comps[1] - comps[0]) > (comps[4] - comps[3]), 'gains must diminish (concavity)'

print(f'BM25 TF component (k1={BM25_K1}, length off), ceiling k1+1 = {ceiling}:')
for f, c in zip((1, 2, 5, 10, 50, 1000), comps):
    print(f'  count {f:>4} -> {c:.4f}')

scores = bm25_scores(full_corpus, QUERY)
order = np.argsort(-scores)
# Same winner as TF-IDF, and the non-matching docs score exactly zero.
assert labels[int(order[0])] == 'D3', 'BM25 should rank D3 first'
assert np.isclose(scores[1], 0.0) and np.isclose(scores[3], 0.0) and np.isclose(scores[4], 0.0), \
    'documents matching neither query term must score 0'

print(f'\nBM25 ranking for {QUERY!r} (k1={BM25_K1}, b={BM25_B}):')
for i in order:
    print(f'  {labels[i]}: {scores[i]:.3f}  {full_corpus[i]!r}')

BM25 TF component (k1=1.5, length off), ceiling k1+1 = 2.5:
  count    1 -> 1.0000
  count    2 -> 1.4286
  count    5 -> 1.9231
  count   10 -> 2.1739
  count   50 -> 2.4272
  count 1000 -> 2.4963

BM25 ranking for 'happy cat' (k1=1.5, b=0.75):
  D3: 2.388  'the happy cat chased the dog'
  D1: 0.924  'the cat sat on the mat'
  D2: 0.000  'the dog sat on the log'
  D4: 0.000  'a quick brown fox jumps over the lazy dog'
  D5: 0.000  'cats and dogs are common household pets'


## 9. The fit-on-test leak — the #1 TF-IDF bug

One last demonstration of the most common TF-IDF mistake (in interviews and real notebooks): **fitting the vectorizer on data that includes the query/test set** leaks document-frequency statistics into the features. We show the cosine score for the query changes when you (wrongly) fold the query into the fit — proof that `fit` must see training data only, then `transform` everything else.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# CORRECT: fit on the corpus, transform the query into that fitted space.
tv_ok = TfidfVectorizer(token_pattern=TOKEN_PATTERN, norm='l2')
X = tv_ok.fit_transform(full_corpus)
q_ok = tv_ok.transform([QUERY])
cos_ok = cosine_similarity(q_ok, X).ravel()

# WRONG: fit on corpus + query together, leaking the query's df into the IDF.
tv_leak = TfidfVectorizer(token_pattern=TOKEN_PATTERN, norm='l2')
X_leak = tv_leak.fit_transform(list(full_corpus) + [QUERY])
cos_leak = cosine_similarity(X_leak[-1], X_leak[:-1]).ravel()

# Our hand implementation matches the CORRECT discipline to machine precision.
assert np.allclose(cos_ok, tfidf_query_scores(full_corpus, QUERY), atol=1e-9), \
    'hand tfidf_query_scores must match correct sklearn fit/transform'
# And the leak actually changes the scores — it is not a harmless habit.
assert not np.allclose(cos_ok, cos_leak), 'fitting on the query leaks df stats and shifts scores'

print('query cosine vs each document:')
print(f"{'doc':<4} {'correct':>9} {'leaked':>9}")
for i in range(len(full_corpus)):
    print(f'{labels[i]:<4} {cos_ok[i]:>9.4f} {cos_leak[i]:>9.4f}')
print('\nThe two columns differ -> fitting on the query is a real leak, not a harmless shortcut.')

query cosine vs each document:
doc    correct    leaked
D1      0.2465    0.2171
D2      0.0000    0.0000
D3      0.6147    0.5340
D4      0.0000    0.0000
D5      0.0000    0.0000

The two columns differ -> fitting on the query is a real leak, not a harmless shortcut.


## Recap

- **Bag-of-words** turns each document into a count vector over a fixed vocabulary, discarding order (cell 1).
- Raw counts over-reward repetition; **concave TF** (log-normalized) fixes it (cell 2).
- **IDF** $=\log(N/df)$ is the surprisal of a term — ubiquitous words carry 0 information (cell 3).
- **TF-IDF** = TF × IDF, L2-normalized, reproduces `scikit-learn` to machine precision (cells 4–5).
- **Cosine** ranks by angle (length-invariant), and TF-IDF **reranks retrieval** better than raw counts by rewarding rare query terms (cells 6–7).
- **BM25** adds TF saturation ($k_1$) and length normalization ($b$) — the production search upgrade (cell 8).
- **Fit on training data only** — folding the query into the fit leaks df statistics (cell 9).

Every number here is also what the concept page and the figures show, because all three import the same `bow_tfidf.py`.